# DriverGuard — обучение YOLOv8n для детектора нарушений (ML-2)

Ноутбук для Google Colab (GPU: Среда выполнения → Сменить среду → T4). Запускает те же скрипты, что и локально: `scripts/prepare_dataset.py` → `scripts/train.py` → `scripts/evaluate.py` → `scripts/export.py`.

Порядок: 1) датасет (zip с папками источников) на Google Drive; 2) выполнить ячейки сверху вниз; 3) скачать `models/violations_vN.onnx` и `models/registry.json` и закоммитить реестр в репозиторий.

In [ ]:
!nvidia-smi -L || echo 'GPU не выбран: Среда выполнения → Сменить среду выполнения → T4 GPU'
!pip -q install ultralytics onnx onnxruntime onnxslim

## 1. Код проекта

Репозиторий приватный: создайте [токен GitHub](https://github.com/settings/tokens) с правом `repo` и вставьте ниже, либо загрузите папку `ml/` вручную (Файлы → Загрузить).

In [ ]:
import os, getpass
TOKEN = getpass.getpass('GitHub token (Enter — пропустить, если ml/ загружен вручную): ')
BRANCH = 'claude/driverguard-full-build-92zul4'
if TOKEN:
    !git clone -q --depth 1 -b $BRANCH https://$TOKEN@github.com/edserver2026-cpu/driverguard1.git /content/driverguard1
    %cd /content/driverguard1/ml
else:
    %cd /content/ml
!pip -q install -e .

## 2. Датасет

Zip с источниками (например `raw/cabin/…`, `raw/roboflow-phone/…`) лежит на Google Drive. Имена кадров собственной съёмки — `cabin__<сессия>__000123.jpg` (см. `scripts/extract_frames.py`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/driverguard/dataset_raw.zip'   # ← путь к вашему архиву
!rm -rf data/raw && mkdir -p data/raw && unzip -q -o $ZIP -d data/raw
!ls data/raw

In [ ]:
# Сплит по сессиям, grayscale-копии для ИК, ворота ML-1 (≥500 изображений на класс, ≥20 % своих кадров)
!python scripts/prepare_dataset.py data/raw/* --out data/dataset --force --no-gate

## 3. Обучение

Ворота выпуска: mAP@50 ≥ 0.85 на валидации. Версия пишется в `models/registry.json` (ML-4).

In [ ]:
VERSION = 'v1'          # следующая версия: v2, v3 …
EPOCHS = 100
!python scripts/train.py --data data/dataset/data.yaml --epochs $EPOCHS --imgsz 640 --batch 32 --version $VERSION --notes 'Colab T4'

In [ ]:
# Матрица ошибок и метрики по классам (ручной разбор — ML-2), тест-видео из кабины по желанию
!python scripts/evaluate.py --weights runs/$VERSION/weights/best.pt --data data/dataset/data.yaml --split test --version $VERSION
from IPython.display import Image, display
display(Image(f'runs/{VERSION}/confusion_matrix.png'))

## 4. Экспорт ONNX + калибровочный набор (ML-3, шаг 1)

In [ ]:
!python scripts/export.py --version $VERSION --calib-from data/dataset/images/val --calib-n 64 --check

In [ ]:
# Забрать артефакты: ONNX, калибровка, реестр — в Drive
!mkdir -p /content/drive/MyDrive/driverguard/models
!cp models/violations_$VERSION.onnx models/registry.json /content/drive/MyDrive/driverguard/models/
!cp -r models/calib_$VERSION models/calib_$VERSION.npy /content/drive/MyDrive/driverguard/models/ 2>/dev/null || true
!python scripts/registry.py list

## 5. На устройстве

Локально: скопировать `models/violations_vN.onnx` и `models/registry.json` в `ml/models/`, затем
```
python scripts/registry.py promote vN
```
— ONNX попадёт в `device/models/`, а `device/config.yaml` получит `detector.model` и `model_version=violations-vN` (ML-4). После приезда Hailo: `hailomz compile …` (команда печатается export.py) и `registry.py set-hef`.